In [ ]:

# ==========================================
# MODULO 3: INICIO DE SESION
# ==========================================

def iniciar_sesion():

    print("\nINICIO DE SESION")
    print("----------------")

    # Verificamos que exista el archivo de usuarios

    if not os.path.exists(archivo_usuarios):

        print("No hay usuarios registrados. Primero debes registrarte.")
        return None

    # Leemos los usuarios guardados en el CSV

    with open(archivo_usuarios, "r", encoding="utf-8") as archivo:

        lector = csv.DictReader(archivo)

        usuarios = list(lector)

    # Solicitamos las credenciales

    usuario = input("Nombre de usuario: ").strip().lower()

    clave = getpass("Contraseña: ")

    # Buscamos el usuario registrado

    for persona in usuarios:

        if persona["usuario"] == usuario:

            # Recuperamos la sal y la contraseña protegida

            sal = bytes.fromhex(persona["sal"])

            clave_hash = hashlib.pbkdf2_hmac(
                "sha256",
                clave.encode(),
                sal,
                100000
            ).hex()

            # Comparamos la contraseña ingresada con la guardada

            if secrets.compare_digest(clave_hash, persona["clave_hash"]):

                print("\n¡INICIO DE SESION EXITOSO!")
                print("Bienvenido(a),", persona["nombre"])

                return persona

            else:

                print("\nContraseña incorrecta.")
                return None

    print("\nEl usuario no existe.")
    return None


In [ ]:

# ==========================================
# MODULO 6: CONFIGURACION DEL SALDO INICIAL
# ==========================================

def configurar_saldo_inicial(usuario_activo):

    # Verificamos que exista una sesion activa

    if usuario_activo is None:
        print("Primero debes iniciar sesion.")
        return

    print("\nCONFIGURACION DEL SALDO INICIAL")
    print("--------------------------------")

    # Consultamos si el archivo ya existe

    registros = []

    if os.path.exists(archivo_saldo_inicial):

        with open(archivo_saldo_inicial, "r", encoding="utf-8") as archivo:

            lector = csv.DictReader(archivo)
            registros = list(lector)

    # Buscamos si el estudiante ya tiene un saldo inicial

    saldo_actual = None

    for registro in registros:

        if registro["id_usuario"] == str(usuario_activo["id_usuario"]):

            saldo_actual = registro["saldo_inicial"]

            break

    if saldo_actual is not None:

        print("Tu saldo inicial actual es: $", format(float(saldo_actual), ",.0f"))
        print("Puedes actualizarlo si es necesario.")

    # Solicitamos el nuevo saldo inicial

    while True:

        try:

            saldo_inicial = int(
                input("\n¿Con cuanto dinero comienzas? $")
            )

            if saldo_inicial < 0:

                print("El saldo no puede ser negativo.")
                continue

            break

        except ValueError:

            print("Debes ingresar un numero entero.")

    # Creamos el nuevo registro

    nuevo_registro = {
        "id_usuario": usuario_activo["id_usuario"],
        "saldo_inicial": saldo_inicial,
        "fecha_actualizacion": date.today().isoformat()
    }

    # Conservamos los saldos de los demas usuarios

    nuevos_registros = []

    for registro in registros:

        if registro["id_usuario"] != str(usuario_activo["id_usuario"]):

            nuevos_registros.append(registro)

    # Agregamos el saldo actualizado del usuario activo

    nuevos_registros.append(nuevo_registro)

    # Guardamos la informacion en el archivo CSV

    campos = [
        "id_usuario",
        "saldo_inicial",
        "fecha_actualizacion"
    ]

    with open(archivo_saldo_inicial, "w", newline="", encoding="utf-8") as archivo:

        escritor = csv.DictWriter(archivo, fieldnames=campos)

        escritor.writeheader()

        escritor.writerows(nuevos_registros)

    print("\n¡SALDO INICIAL CONFIGURADO CORRECTAMENTE!")
    print("-----------------------------------------")
    print("Saldo inicial: $", format(saldo_inicial, ",.0f"))